# Clickbait: LLM level (Qwen2.5-7B-Instruct, 4-bit, Kaggle GPU)
Three prompting modes on the VnExpress dev headlines: zero-shot, few-shot (8 examples from the
VnExpress part of ViClickbait-2025, never from dev) and few-shot with Chain-of-Thought.
The score is P("1") / (P("0") + P("1")) for the label token, so PR-AUC can be computed.

Input: the same private Kaggle dataset as the PhoBERT notebook. Settings: GPU T4 x2 or P100, Internet on.
Output (put in `results/clickbait/kaggle/`): `dev_scores_qwen_zero.csv`, `dev_scores_qwen_few.csv`,
`dev_scores_qwen_cot.csv`, and `qwen_cot_outputs.jsonl` (reasoning text, for error analysis).

In [ ]:
!pip install -q -U bitsandbytes accelerate

In [ ]:
import glob, json, os
import numpy as np, pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DATA = os.path.dirname(glob.glob("/kaggle/input/**/viclickbait.csv", recursive=True)[0])
OUT = "/kaggle/working"
MODEL = "Qwen/Qwen2.5-7B-Instruct"
SEED = 42
torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_quant_type="nf4"))
model.eval()
dev = pd.read_csv(f"{DATA}/dev_headlines.csv", dtype={"id": str})
vc = pd.read_csv(f"{DATA}/viclickbait.csv")
ID0, ID1 = tok.encode("0", add_special_tokens=False)[0], tok.encode("1", add_special_tokens=False)[0]

In [ ]:
DEFINITION = """Bạn là người gán nhãn tiêu đề báo tiếng Việt. Tiêu đề là clickbait (1) nếu có ít nhất một dấu hiệu:
1. Tạo khoảng trống tò mò: cố ý giấu thông tin chính (đại từ mơ hồ như "người này", "điều đó", "lý do bất ngờ"; câu hỏi gợi tò mò mà tiêu đề không trả lời; danh sách không nói rõ nội dung).
2. Từ ngữ cảm xúc hoặc cường điệu do người viết chọn để gây ấn tượng ("gây sốc", "chấn động", "rúng động", ẩn dụ giật gân, biệt danh giật gân thay cho tên thật).
3. Bỏ sót chi tiết then chốt khiến tiêu đề dễ gây hiểu lầm.
Không clickbait (0): tiêu đề cung cấp thông tin, khách quan, có đủ ngữ cảnh. Từ mạnh nhưng mô tả sự kiện ("bắt", "tử vong", "khởi tố") không phải cường điệu. Trích dẫn có nêu người nói, tên tác phẩm trong ngoặc kép, câu hỏi chính sách hoặc hướng dẫn thật đều là 0."""

shots = pd.concat([vc[(vc.source == "VnExpress") & (vc.y == 1)].sample(4, random_state=SEED),
                   vc[(vc.source == "VnExpress") & (vc.y == 0)].sample(4, random_state=SEED)]).sample(frac=1, random_state=SEED)
SHOTS = "\n".join(f"Tiêu đề: {r.title}\nNhãn: {r.y}" for r in shots.itertuples())

def messages(title, mode):
    task = DEFINITION
    if mode in ("few", "cot"):
        task += "\n\nVí dụ:\n" + SHOTS
    if mode == "cot":
        task += "\n\nHãy xét lần lượt ba dấu hiệu trong 2 đến 3 câu ngắn, rồi kết thúc bằng một dòng 'Nhãn: 0' hoặc 'Nhãn: 1'."
        user = f"Tiêu đề: {title}"
    else:
        task += "\n\nChỉ trả lời một chữ số 0 hoặc 1."
        user = f"Tiêu đề: {title}\nNhãn:"
    return [{"role": "system", "content": task}, {"role": "user", "content": user}]

@torch.no_grad()
def label_prob(prefix_ids):
    logits = model(prefix_ids).logits[0, -1]
    p = torch.softmax(logits[[ID0, ID1]].float(), -1)
    return float(p[1])

@torch.no_grad()
def score(title, mode):
    ids = tok.apply_chat_template(messages(title, mode), add_generation_prompt=True, return_tensors="pt").to(model.device)
    if mode != "cot":
        return label_prob(ids), None
    gen = model.generate(ids, max_new_tokens=200, do_sample=False)
    text = tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)
    reasoning = text.split("Nhãn:")[0].rstrip()
    prefix = tok(reasoning + "\nNhãn: ", add_special_tokens=False, return_tensors="pt").input_ids.to(model.device)
    return label_prob(torch.cat([ids, prefix], dim=1)), text

In [ ]:
for mode in ("zero", "few", "cot"):
    scores, texts = [], []
    for t in dev.title:
        s, txt = score(t, mode)
        scores.append(s); texts.append(txt)
    s = np.array(scores)
    pd.DataFrame({"id": dev["id"], "score": s.round(4), "pred": (s >= 0.5).astype(int)}).to_csv(
        f"{OUT}/dev_scores_qwen_{mode}.csv", index=False)
    if mode == "cot":
        with open(f"{OUT}/qwen_cot_outputs.jsonl", "w") as f:
            for i, txt in zip(dev["id"], texts):
                f.write(json.dumps({"id": i, "output": txt}, ensure_ascii=False) + "\n")
    print(mode, "predicted share", (s >= 0.5).mean())